In [ ]:
pip install --upgrade snowflake-connector-python

In [ ]:
!python --version


In [ ]:
import pandas as pd
from snowflake.snowpark.context import get_active_session
import matplotlib as mplt
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

session = get_active_session()

In [ ]:
#session.sql("use database walmartdb")
session.sql("use schema w_report")

In [ ]:
session.sql("select current_database(), current_schema(), current_warehouse()").show()

In [ ]:
df_store_holiday = session.sql("""
                                select * from WALMARTDB.W_REPORT.WEEKLYSALESBYSTORE"""
                            ).to_pandas()

df_store_holiday.head()

In [ ]:
plt.figure(figsize=(16,8))

ax = sns.barplot(
    data= df_store_holiday,
    x="STORE_ID",
    y="SUM_WEEKLY_SALES",
    hue="ISHOLIDAY",
    palette={False:"#2b82c9",True: "#555555"}
)
for container in ax.containers:
    labels = [
        f"{int(v.get_height()/1e6)}M" for v in container
    ]
    ax.bar_label(
        container,
        labels=labels,
        #padding=3,  # Distance above bar
        fontsize=8,
        rotation=45 # Rotates labels 90 deg to avoid overlap
    )
# Format Y-axis to Millions
ax.yaxis.set_major_formatter(lambda x, pos: f"{int(x/1e6)}M")

plt.title("Sales by Store and isHoliday" , fontsize=14)
plt.xlabel("Store ID")
plt.ylabel("Sales")
plt.legend(title="IsHoliday", frameon=False)
plt.show()

In [ ]:
df_holiday2 = df_store_holiday.groupby("ISHOLIDAY", as_index=False)["SUM_WEEKLY_SALES"].sum()
df_holiday2.head()

In [ ]:
df_holiday = df_store_holiday.groupby("ISHOLIDAY")["SUM_WEEKLY_SALES"].sum()

plt.figure(figsize=(7,7))
plt.pie(
    df_holiday,
    labels=["False", "True"],
    colors=["#2b82c9","#555555"],
    startangle=120,
    explode=(0,0.1),
)
plt.title("Weekly Sales by Holiday")
plt.legend(title="IsHoliday")
plt.show()

In [ ]:
fig = px.pie(
    df_holiday2,
    values="SUM_WEEKLY_SALES",
    names="ISHOLIDAY",
    title="Weekly Sales by Holiday",
    color="ISHOLIDAY",
    color_discrete_map={False:'#2b82c9', True:"#555555"},
    
)
fig.update_traces(textinfo="label", textposition="outside",)
#fig.update_traces(textinfo="percent+label", hoverinfo="value")
fig.show()

In [ ]:
# Pie chart for Top 10 Stores by Sales
top_10 = (
    df_store_holiday.groupby("STORE_ID")["SUM_WEEKLY_SALES"]
    .sum()
    .nlargest(10)
)

plt.figure(figsize=(8, 8))
plt.pie(
    top_10,
    labels=[f"Store {i}" for i in top_10.index],
   # autopct="%1.1f%%",
   # startangle=140,
)
plt.title("Top 10 Stores Sales Contribution")
plt.show()

In [ ]:
df_tempyear = session.sql("""
                                select * from WALMARTDB.W_REPORT.WEEKLYSALESBYTEMPANDYEAR"""
                            ).to_pandas()

df_tempyear.head()

In [ ]:

plt.figure(figsize=(17,7))

df_tempyear["YEARS"] = df_tempyear["YEARS"].astype(str)
df_tempyear2=df_tempyear.head(25)

ax= sns.barplot(
    data= df_tempyear2,
    x="STORE_TEMPERATURE",
    y="SUM_WEEKLY_SALES",
    hue="YEARS",
)
    #format y axes in M
ax.yaxis.set_major_formatter(lambda x , pos:f"{x/1e6:.1f}M")

plt.show()

In [ ]:
df_tempyear["SALES_DIFF"] = df_tempyear["SUM_WEEKLY_SALES"].diff().fillna(df_tempyear["SUM_WEEKLY_SALES"])
df_tempyear.head()

In [ ]:
x_labels = (df_tempyear2["STORE_TEMPERATURE"].astype(str) + " (" + df_tempyear2["YEARS"] + ")")

fig = go.Figure(
    go.Waterfall(
        name="Weekly Sales",
        orientation="v",
        measure=["relative"] * len(df_tempyear2),
        x=x_labels,
        textposition="outside",
        text= [f"{v/1e6:+.2f}M" for v in df_tempyear2["SALES_DIFF"]],
        y= df_tempyear2["SALES_DIFF"],
        increasing={"marker":{"color":"#2b82c9"}},
        decreasing={"marker":{"color":"#555555"}},
        connector={"line":{"color":"#aaaaaa"}},
    )
)
fig.update_layout(
    title="Weekly Sales by Temperature and Years",
    xaxis_title="Temperature - Years",
    yaxis_title="Weekly Sales",
    waterfallgap=0.3,
)

fig.show()

In [ ]:
df_piv = df_tempyear.pivot(
index="STORE_TEMPERATURE", columns="YEARS", values="SUM_WEEKLY_SALES").reset_index()

df_piv.head()

In [ ]:
df_flat = df_piv.melt(id_vars=['STORE_TEMPERATURE'], var_name="YEARS", value_name="SALES").fillna(0).sort_values(by=["STORE_TEMPERATURE","YEARS"]).reset_index(drop=False).head(50)


In [ ]:
print(df_flat)

In [ ]:
df_flat["SALES_DIFF"] = df_flat.groupby("STORE_TEMPERATURE")["SALES"].diff().fillna(df_flat["SALES"])
print(df_flat)

In [ ]:

df_sorted = df_flat
# 2. Create a unique categorical label string for every row (e.g., "-2.06 \n 2011")
df_sorted["X_LABEL"] = (
    df_sorted["STORE_TEMPERATURE"].astype(str)
    + "<br>"
    + df_sorted["YEARS"].astype(str)
)

# 3. Calculate step-by-step relative changes
df_sorted["SALES_DIFF"] = df_sorted["SALES"].diff().fillna(
    df_sorted["SALES"]
)

fig = go.Figure(
    go.Waterfall(
        name="Weekly Sales",
        orientation="v",
        measure=["relative"] * len(df_sorted),
        x=df_sorted["X_LABEL"],
        y=df_sorted["SALES_DIFF"],
        text=[f"{v/1e6:+.2f}M" for v in df_sorted["SALES_DIFF"]],
        textposition="outside",
        increasing={"marker": {"color": "#2b82c9"}},  #  Blue
        decreasing={"marker": {"color": "#555555"}},  #  Grey
        connector={"line": {"color": "#aaaaaa", "width": 1}},
    )
)

# 4. Force Plotly to treat the X-axis strictly as discrete categories (not numbers)
fig.update_xaxes(type="category", title="Temperature & Year")
fig.update_yaxes(title="Weekly Sales", tickformat="~s")

fig.update_layout(
    title="Weekly_Sales by Temperature and Year",
    waterfallgap=0.2,
    showlegend=False,
)

fig.show()

In [ ]:
df_storesize = session.sql("""
                                select * from WALMARTDB.W_REPORT.WEEKLYSALEBYSTORESIZE"""
                            ).to_pandas()

df_storesize= df_storesize.sort_values(by="STORE_SIZE")
print(df_storesize)

In [ ]:
plt.figure(figsize=(12,8))
sns.lineplot(
    data=df_storesize,
    x="STORE_SIZE",
    y="SUM_WEEKLY_SALES",
    color="#2b82c9",
    linewidth=1,
)
plt.fill_between(
    df_storesize["STORE_SIZE"],
    df_storesize["SUM_WEEKLY_SALES"],
    color="#2b82c9",
    alpha=0.2,
)
ax = plt.gca()
ax.xaxis.set_major_formatter(lambda x, pos:f"{x/1e3:.0f}K")
ax.yaxis.set_major_formatter(lambda y, pos:f"{y/1e6:.0f}M")

plt.title("Weekly Sales by Store Size")
plt.xlabel("Store Size")
plt.ylabel("Weekly Sales")
#plt.tight_layout()

plt.show()

In [ ]:
fig = px.area(
    df_storesize, 
    x="STORE_SIZE",
    y="SUM_WEEKLY_SALES",
    hover_data=["STORE_ID"],
    title="Weekly Sales by Size",
    
    
)
fig.update_traces( line_color="#2b82c9", fillcolor="rgba(75, 156, 211, 0.2)")
fig.show()

In [ ]:


df_typebymonth = session.sql("""
                                select * from WALMARTDB.W_REPORT.WEEKLYSALEBYSTORETYPEANDMONTH"""
                            ).to_pandas()

df_typebymonth.head()

In [ ]:
import calendar

# 1. Map month numbers (1-12) to month names ('January', 'February', ...)
month_map = {i: calendar.month_name[i] for i in range(1, 13)}
df_typebymonth["MONTH_NAME"] = df_typebymonth["MONTHS"].map(month_map)

# 2. Sort by month number so the x-axis follows calendar order
df_sorted = df_typebymonth.sort_values("MONTHS").copy()

# 3. Create formatted text labels in Millions (e.g., '185M')
df_sorted["TEXT_LABEL"] = (df_sorted["SUM_WEEKLY_SALES"] / 1e6).round(0).astype(int).astype(str) + "M"

# 4. Build line chart
fig = px.line(
    df_sorted,
    x="MONTH_NAME",
    y="SUM_WEEKLY_SALES",
    color="STORE_TYPE",
    text="TEXT_LABEL",
    title="Weekly_Sales by Month and Type",
    color_discrete_map={
        "A": "#2b82c9", 
        "B": "#555555", 
        "C": "#ff5a52",  
    },
    labels={
        "MONTH_NAME": "Month",
        "SUM_WEEKLY_SALES": "Weekly Sales",
        "STORE_TYPE": "Type",
    },
)

# Position labels above the line markers and match the style
fig.update_traces(textposition="top center")
fig.update_xaxes(type="category")
fig.update_yaxes(ticksuffix="M", tickformat="~s")

fig.show()

In [ ]:

# Ensure month names and proper chronological ordering
month_order = list(calendar.month_name)[1:]
df_typebymonth["MONTH_NAME"] = df_typebymonth["MONTHS"].apply(
    lambda x: calendar.month_name[int(x)]
)

plt.figure(figsize=(14, 6))

ax = sns.lineplot(
    data=df_typebymonth,
    x="MONTH_NAME",
    y="SUM_WEEKLY_SALES",
    hue="STORE_TYPE",
    palette={"A": "#2b82c9", "B": "#555555", "C": "#ff5a52"},
    marker="o",
)

# Annotate each point with values formatted in Millions
for row in df_typebymonth.itertuples():
    val_m = f"{int(row.SUM_WEEKLY_SALES / 1e6)}M"
    # Find position on x-axis based on month index
    x_pos = month_order.index(row.MONTH_NAME)
    ax.annotate(
        val_m,
        (x_pos, row.SUM_WEEKLY_SALES),
        textcoords="offset points",
        xytext=(0, 6),
        ha="center",
        fontsize=8,
    )

ax.yaxis.set_major_formatter(lambda y, pos: f"{int(y/1e6)}M")
plt.title("Weekly_Sales by Month and Type", loc="left", fontsize=14)
plt.xlabel("")
plt.ylabel("Weekly Sales")
plt.legend(title="Type", frameon=False)
plt.tight_layout()
plt.show()

In [ ]:
df_markdown = session.sql("""
                                select * from WALMARTDB.W_REPORT.MARKDOWNSALESBYYEARANDSTORE"""
                            ).to_pandas()

df_markdown.head()

In [ ]:
# 1. Melt the Markdown columns from wide format to long format
df_melted = df_markdown.melt(
    id_vars=["YEARS"],
    value_vars=[
        "MARKDOWN1",
        "MARKDOWN2",
        "MARKDOWN3",
        "MARKDOWN4",
        "MARKDOWN5",
    ],
    var_name="MARKDOWN_TYPE",
    value_name="AMOUNT",
)

# 2. Aggregate the total amounts by Year and Markdown Type
df_grouped = (
    df_melted.groupby(["YEARS", "MARKDOWN_TYPE"], as_index=False)["AMOUNT"]
    .sum()
    .fillna(0)
)

# 3. Create formatted data labels in Billions (e.g., '0.98bn')
df_grouped["LABEL"] = (df_grouped["AMOUNT"] / 1e9).round(2).astype(
    str
) + "bn"

fig = px.bar(
    df_grouped,
    x="YEARS",
    y="AMOUNT",
    color="MARKDOWN_TYPE",
    barmode="group",
    text="LABEL",
    title="MarkDown1, MarkDown2, MarkDown3, MarkDown4 and MarkDown5 by Year",
    color_discrete_map={
        "MARKDOWN1": "#4285F4",  # Blue
        "MARKDOWN2": "#3c4043",  # Dark Grey
        "MARKDOWN3": "#ea4335",  # Red
        "MARKDOWN4": "#a5673f",  # Brown
        "MARKDOWN5": "#5f6368",  # Slate Grey
    },
    labels={
        "YEARS": "",
        "AMOUNT": "",
        "MARKDOWN_TYPE": "",
    },
)

# Position labels above bars and format Y-axis in Billions
fig.update_traces(textposition="outside", textfont_size=9)
fig.update_xaxes(type="category")
fig.update_yaxes(ticksuffix="bn", tickformat="~s")

fig.show()

In [ ]:
# 1. Melt columns into long format
df_melted = df_markdown.melt(
    id_vars=["YEARS"],
    value_vars=[
        "MARKDOWN1",
        "MARKDOWN2",
        "MARKDOWN3",
        "MARKDOWN4",
        "MARKDOWN5",
    ],
    var_name="MARKDOWN_TYPE",
    value_name="AMOUNT",
)

# 2. Aggregate sum by Year and Markdown Type
df_grouped = (
    df_melted.groupby(["YEARS", "MARKDOWN_TYPE"], as_index=False)["AMOUNT"]
    .sum()
    .fillna(0)
)

plt.figure(figsize=(10, 6))

# 3. Plot grouped bar chart
ax = sns.barplot(
    data=df_grouped,
    x="YEARS",
    y="AMOUNT",
    hue="MARKDOWN_TYPE",
    palette=["#4285F4", "#3c4043", "#ea4335", "#a5673f", "#5f6368"],
)

# 4. Add floating text labels in Billions above each bar
for container in ax.containers:
    labels = [f"{v.get_height()/1e9:.2f}bn" for v in container]
    ax.bar_label(container, labels=labels, padding=3, fontsize=8)

# Format Y-axis to display in Billions
ax.yaxis.set_major_formatter(lambda x, pos: f"{x/1e9:.1f}bn")

plt.title("MarkDown1, MarkDown2, MarkDown3, MarkDown4 and MarkDown5 by Year")
plt.xlabel("")
plt.ylabel("")
plt.legend(frameon=False)
plt.tight_layout()
plt.show()

In [ ]:
df_storetype = session.sql("""
                                select * from WALMARTDB.W_REPORT.WEEKLYSALESBYSTORETYPE"""
                            ).to_pandas()

df_storetype.head()

In [ ]:
# 1. Format text labels in Millions (e.g., '222M')
df_storetype["LABEL"] = (
    (df_storetype["SUM_WEEKLY_SALES"] / 1e6).round(0).astype(int).astype(str)
    + "M"
)

# 2. Build horizontal bar chart grouped by STORE_TYPE
fig = px.bar(
    df_storetype,
    y="STORE_TYPE",
    x="SUM_WEEKLY_SALES",
    color="STORE_ID",
    text="LABEL",
    orientation="h",
    barmode="group",
    title="Weekly_Sales by Type and Store",
    labels={
        "STORE_TYPE": "Type",
        "SUM_WEEKLY_SALES": "Weekly Sales",
        "STORE_ID": "Store",
    },
)

# Position text at the end of each bar
fig.update_traces(textposition="outside", textfont_size=9)
fig.update_xaxes(ticksuffix="M", tickformat="~s")

fig.show()

In [ ]:
plt.figure(figsize=(12, 8))

# Horizontal grouped bar chart
ax = sns.barplot(
    data=df_storetype,
    y="STORE_TYPE",
    x="SUM_WEEKLY_SALES",
    hue="STORE_ID",
    orient="h",
)

# Add data labels outside each bar formatted in Millions
for container in ax.containers:
    labels = [
        f"{int(v.get_width()/1e6)}M" if v.get_width() > 0 else ""
        for v in container
    ]
    ax.bar_label(container, labels=labels, padding=3, fontsize=7)

# Format X-axis in Millions
ax.xaxis.set_major_formatter(lambda x, pos: f"{int(x/1e6)}M")

plt.title("Weekly_Sales by Type and Store", loc="left", fontsize=14)
plt.ylabel("Type")
plt.xlabel("Weekly Sales")
plt.legend(
    title="Store",
    bbox_to_anchor=(1.05, 1),
    loc="upper left",
    ncol=2,
    frameon=False,
)

plt.tight_layout()
plt.show()

In [ ]:
# 1. Aggregate sales by STORE_TYPE
df_pie = (
    df_storetype.groupby("STORE_TYPE", as_index=False)["SUM_WEEKLY_SALES"]
    .sum()
)

# 2. Build Pie Chart
fig = px.pie(
    df_pie,
    values="SUM_WEEKLY_SALES",
    names="STORE_TYPE",
    title="Weekly_Sales by store Type",
    color="STORE_TYPE",
    color_discrete_map={
        "A": "#70a1ff",  # Light Blue
        "B": "#2f3542",  # Dark Slate Grey
        "C": "#ff6b81",  # Red / Coral
    },
)

# 3. Match layout: slice callout lines and custom legend title
fig.update_traces(textposition="outside", textinfo="label")
fig.update_layout(
    legend_title_text="Type",
    showlegend=True
)

fig.show()

In [ ]:
# 1. Aggregate sales by STORE_TYPE
df_pie = (
    df_storetype.groupby("STORE_TYPE")["SUM_WEEKLY_SALES"]
    .sum()
)

colors = {"A": "#70a1ff", "B": "#2f3542", "C": "#ff6b81"}

# 2. Plot Pie Chart
plt.figure(figsize=(7, 6))
plt.pie(
    df_pie.values,
    labels=df_pie.index,
    colors=[colors[k] for k in df_pie.index],
    startangle=90,
    counterclock=False,
)

plt.title("Weekly_Sales by store Type", loc="left", fontsize=12)
plt.legend(
    title="Type",
    labels=df_pie.index,
    loc="upper left",
    frameon=False,
)

plt.tight_layout()
plt.show()

In [ ]:
df_fuelyear = session.sql("""
                                select * from WALMARTDB.W_REPORT.FUELPRICEANDYEAR"""
                            ).to_pandas()

df_fuelyear.head()

In [ ]:
# 1. Aggregate total FUEL_PRICE by YEARS
df_donut = (
    df_fuelyear.groupby("YEARS", as_index=False)["FUEL_PRICE"]
    .sum()
)

# 2. Convert YEARS to string for categorical mapping
df_donut["YEARS"] = df_donut["YEARS"].astype(str)

# 3. Create Donut Chart (Pie chart with a hole)
fig = px.pie(
    df_donut,
    values="FUEL_PRICE",
    names="YEARS",
    title="Fuel_Price by Year",
    hole=0.55,  # Creates the donut shape
    color="YEARS",
    color_discrete_map={
        "2010": "#3b82f6",  # Bright Blue
        "2011": "#5b9bd5",  # Medium Teal/Blue
        "2012": "#385764",  # Dark Slate
    },
)

# 4. Position callout labels on the outside
fig.update_traces(textposition="outside", textinfo="label")
fig.update_layout(showlegend=False)

fig.show()

In [ ]:
# 1. Aggregate total FUEL_PRICE by YEARS
df_donut = df_fuelyear.groupby("YEARS")["FUEL_PRICE"].sum()

colors = ["#3b82f6", "#5b9bd5", "#385764"]

plt.figure(figsize=(6, 6))

# 2. Plot Donut Chart using wedgeprops
plt.pie(
    df_donut.values,
    labels=[str(y) for y in df_donut.index],
    colors=colors,
    startangle=90,
    counterclock=False,
    wedgeprops=dict(width=0.45, edgecolor="w"),
)

plt.title("Fuel_Price by Year", loc="left", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
df_dategrain = session.sql("""
                                select * from WALMARTDB.W_REPORT.WEEKLYSALESDATEGRAIN"""
                            ).to_pandas()

df_dategrain.head()

In [ ]:
import calendar
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Prepare monthly ordering and labels
month_names = list(calendar.month_name)[1:]

# 1. Aggregate datasets
df_year = (
    df_dategrain.groupby("YEARS", as_index=False)["SUM_WEEKLY_SALES"]
    .sum()
    .sort_values("YEARS")
)
df_month = (
    df_dategrain.groupby("MONTHS", as_index=False)["SUM_WEEKLY_SALES"]
    .sum()
    .sort_values("MONTHS")
)
df_month["MONTH_NAME"] = df_month["MONTHS"].apply(
    lambda x: calendar.month_name[int(x)]
)
df_day = (
    df_dategrain.groupby("DAYSNUM", as_index=False)["SUM_WEEKLY_SALES"]
    .sum()
    .sort_values("DAYSNUM")
)

# 2. Setup Subplot Canvas (Row 1: Year & Month, Row 2: Day)
fig = make_subplots(
    rows=2,
    cols=2,
    specs=[[{"type": "bar"}, {"type": "bar"}], [{"colspan": 2}, None]],
    subplot_titles=(
        "Weekly_Sales by Year",
        "Weekly_Sales by Month",
        "Weekly_Sales by Day",
    ),
)

# --- Chart 1: Sales by Year ---
fig.add_trace(
    go.Bar(
        x=df_year["YEARS"],
        y=df_year["SUM_WEEKLY_SALES"],
        text=(df_year["SUM_WEEKLY_SALES"] / 1e9).round(2).astype(str) + "bn",
        textposition="outside",
        marker_color="#00a8ff",
        showlegend=False,
    ),
    row=1,
    col=1,
)

# --- Chart 2: Sales by Month ---
fig.add_trace(
    go.Bar(
        x=df_month["MONTH_NAME"],
        y=df_month["SUM_WEEKLY_SALES"],
        text=(df_month["SUM_WEEKLY_SALES"] / 1e9).round(2).astype(str) + "bn",
        textposition="outside",
        marker_color="#00a8ff",
        showlegend=False,
    ),
    row=1,
    col=2,
)

# --- Chart 3: Sales by Day ---
fig.add_trace(
    go.Bar(
        x=df_day["DAYSNUM"],
        y=df_day["SUM_WEEKLY_SALES"],
        text=(df_day["SUM_WEEKLY_SALES"] / 1e6).round(0).astype(int).astype(
            str
        )
        + "M",
        textposition="outside",
        marker_color="#00a8ff",
        showlegend=False,
    ),
    row=2,
    col=1,
)

# Formatting axes
fig.update_xaxes(type="category", row=1, col=1)
fig.update_xaxes(categoryorder="array", categoryarray=month_names, row=1, col=2)
fig.update_xaxes(dtick=5, row=2, col=1)

fig.update_yaxes(ticksuffix="bn", tickformat="~s", row=1, col=1)
fig.update_yaxes(ticksuffix="bn", tickformat="~s", row=1, col=2)
fig.update_yaxes(ticksuffix="M", tickformat="~s", row=2, col=1)

fig.update_layout(
    height=700,
    title_text="weekly sales by year, month and date",
    title_x=0.5,
)
fig.show()

In [ ]:
import calendar
import matplotlib.pyplot as plt
import seaborn as sns

# Aggregate data
df_year = (
    df_dategrain.groupby("YEARS", as_index=False)["SUM_WEEKLY_SALES"].sum()
)
df_month = (
    df_dategrain.groupby("MONTHS", as_index=False)["SUM_WEEKLY_SALES"].sum()
)
df_month["MONTH_NAME"] = df_month["MONTHS"].apply(
    lambda x: calendar.month_name[int(x)]
)
df_day = (
    df_dategrain.groupby("DAYSNUM", as_index=False)["SUM_WEEKLY_SALES"].sum()
)

fig = plt.figure(figsize=(14, 8))
fig.suptitle(
    "weekly sales by year, month and date", fontsize=16, y=0.98
)

# 1. Sales by Year
ax1 = plt.subplot(2, 2, 1)
sns.barplot(
    data=df_year,
    x="YEARS",
    y="SUM_WEEKLY_SALES",
    color="#00a8ff",
    ax=ax1,
)
ax1.set_title("Weekly_Sales by Year", loc="left")
ax1.yaxis.set_major_formatter(lambda x, pos: f"{x/1e9:.1f}bn")
for p in ax1.patches:
    ax1.annotate(
        f"{p.get_height()/1e9:.2f}bn",
        (p.get_x() + p.get_width() / 2.0, p.get_height()),
        ha="center",
        va="bottom",
        fontsize=8,
    )

# 2. Sales by Month
ax2 = plt.subplot(2, 2, 2)
sns.barplot(
    data=df_month,
    x="MONTH_NAME",
    y="SUM_WEEKLY_SALES",
    color="#00a8ff",
    ax=ax2,
)
ax2.set_title("Weekly_Sales by Month", loc="left")
ax2.tick_params(axis="x", rotation=45)
ax2.yaxis.set_major_formatter(lambda x, pos: f"{x/1e9:.1f}bn")
for p in ax2.patches:
    ax2.annotate(
        f"{p.get_height()/1e9:.2f}bn",
        (p.get_x() + p.get_width() / 2.0, p.get_height()),
        ha="center",
        va="bottom",
        fontsize=8,
    )

# 3. Sales by Day
ax3 = plt.subplot(2, 1, 2)
sns.barplot(
    data=df_day, x="DAYSNUM", y="SUM_WEEKLY_SALES", color="#00a8ff", ax=ax3
)
ax3.set_title("Weekly_Sales by Day", loc="left")
ax3.yaxis.set_major_formatter(lambda x, pos: f"{int(x/1e6)}M")
for p in ax3.patches:
    if p.get_height() > 0:
        ax3.annotate(
            f"{int(p.get_height()/1e6)}M",
            (p.get_x() + p.get_width() / 2.0, p.get_height()),
            ha="center",
            va="bottom",
            fontsize=7,
        )

plt.tight_layout()
plt.show()

In [ ]:
df_cpi = session.sql("""
                                select * from WALMARTDB.W_REPORT.WEEKLYSALESBYCPI"""
                            ).to_pandas()

df_cpi.head()

In [ ]:
import plotly.express as px

# 1. Sort by CPI for proper line plotting
df_sorted = df_cpi.sort_values("CPI").copy()

# 2. Build the dotted line chart
fig = px.line(
    df_sorted,
    x="CPI",
    y="SUM_WEEKLY_SALES",
    title="Weekly_Sales by CPI",
    labels={"CPI": "CPI", "SUM_WEEKLY_SALES": "Weekly Sales"},
)

# 3. Match visual styling (dashed/dotted line)
fig.update_traces(line=dict(dash="dot", color="#2b82c9", width=1.5))

# 4. Format y-axis ticks in Millions (e.g., '14M', '12M')
fig.update_yaxes(ticksuffix="M", tickformat="~s")

fig.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Sort by CPI
df_sorted = df_cpi.sort_values("CPI")

plt.figure(figsize=(12, 6))

# Plot dotted line chart
plt.plot(
    df_sorted["CPI"],
    df_sorted["SUM_WEEKLY_SALES"],
    linestyle=":",
    color="#2b82c9",
    linewidth=1.2,
)

# Format y-axis values in Millions
ax = plt.gca()
ax.yaxis.set_major_formatter(lambda y, pos: f"{int(y/1e6)}M")

plt.title("Weekly_Sales by CPI", loc="left", fontsize=12)
plt.xlabel("CPI")
plt.ylabel("Weekly Sales")
plt.grid(axis="y", linestyle="-", alpha=0.2)

plt.tight_layout()
plt.show()

In [ ]:
df_dept = session.sql("""
                                select * from WALMARTDB.W_REPORT.WEEKLYSALESBYDEPARTMENT"""
                            ).to_pandas()

df_dept.head()

In [ ]:
# Aggregate total sales per department and get top 5
df_top5 = (
    df_dept.groupby("DEPT_ID", as_index=False)["SUM_WEEKLY_SALES"]
    .sum()
    .sort_values(by="SUM_WEEKLY_SALES", ascending=False)
    .head(5)
)

# Format sales for cleaner output
df_top5["FORMATTED_SALES"] = df_top5["SUM_WEEKLY_SALES"].map("{:,.2f}".format)
print(df_top5[["DEPT_ID", "FORMATTED_SALES"]].to_string(index=False))

In [ ]:
import plotly.express as px

# 1. Aggregate and sort by department ID
df_dept_summary = (
    df_dept.groupby("DEPT_ID", as_index=False)["SUM_WEEKLY_SALES"]
    .sum()
    .sort_values("DEPT_ID")
)

# 2. Convert DEPT_ID to string so each department gets a unique color
df_dept_summary["DEPT_ID_STR"] = df_dept_summary["DEPT_ID"].astype(str)

# 3. Create vertical bar chart
fig = px.bar(
    df_dept_summary,
    x="DEPT_ID",
    y="SUM_WEEKLY_SALES",
    color="DEPT_ID_STR",
    title="Weekly_Sales by Dept",
    labels={"DEPT_ID": "Dept", "SUM_WEEKLY_SALES": "Weekly Sales"},
)

# 4. Y-axis in Billions to match original chart
fig.update_yaxes(ticksuffix="bn", tickformat="~s")
fig.update_layout(showlegend=False, bargap=0.1)

fig.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

df_dept_summary = (
    df_dept.groupby("DEPT_ID", as_index=False)["SUM_WEEKLY_SALES"]
    .sum()
    .sort_values("DEPT_ID")
)

plt.figure(figsize=(14, 5))

# Plot bar chart with distinct hues per department
ax = sns.barplot(
    data=df_dept_summary,
    x="DEPT_ID",
    y="SUM_WEEKLY_SALES",
    hue="DEPT_ID",
    palette="tab20",
    legend=False,
)

# Format y-axis values in Billions
ax.yaxis.set_major_formatter(lambda y, pos: f"{y/1e9:.1f}bn")

plt.title("Weekly_Sales by Dept", loc="left", fontsize=12)
plt.xlabel("")
plt.ylabel("")
plt.xticks(rotation=90, fontsize=8)
plt.tight_layout()
plt.show()